In [1]:

%sql
create connection if not exists youtube_earthquake_conn
type HTTP
OPTIONS (
  host = 'https://earthquake.usgs.gov',
  port = 443,
  base_path = '/earthquakes/feed/v1.0/',
  bearer_token = 'na'
)

""


In [2]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

conn = w.connections.get("youtube_earthquake_conn")

base_url = f"{conn.options['host']}:{conn.options['port']}{conn.options['base_path']}"  
print(base_url)

https://earthquake.usgs.gov:443/earthquakes/feed/v1.0/


In [6]:
dbutils.widgets.text("catalog_name", "restapiproj", "Catalog")
dbutils.widgets.text("schema_name", "bronze", "schema")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")

print(f"Catalog Name: {catalog_name}")
print(f"Schema Name: {schema_name}")

Box(children=(Label(value='Catalog'), Text(value='restapiproj')))

Box(children=(Label(value='schema'), Text(value='bronze')))

Catalog Name: restapiproj
Schema Name: bronze


In [7]:
spark.sql(f"use catalog {catalog_name}");
spark.sql(f"use schema {schema_name}");
spark.sql("create volume if not exists youtube_earthquake_volume");

In [8]:
import requests
import json
import datetime
url = f"{base_url}summary/all_hour.geojson"
response = requests.get(url)
if response.status_code != 200:
    raise ValueError(f"Failed to retrieve data: {response.status_code}")
data = response.json()
current_date = datetime.datetime.now().strftime("%Y-%m-%d")
dbutils.fs.put(
    f"/Volumes/{catalog_name}/{schema_name}/youtube_earthquake_volume/earthquake_data_{current_date}.json", 
    json.dumps(data), 
    overwrite=True)


True